# Apache Avro - Rust

All 2 Rust examples from [docs/avro.md](https://platob.github.io/yggdryl/avro/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and expect the
[evcxr](https://github.com/evcxr/evcxr) kernel. Declare the crate once, in
a cell of your own, before running them:

```rust
:dep yggdryl = { version = "0.1", features = ["parquet", "iceberg"] }
```

In [ ]:
use yggdryl::io::{Buffer, IOBase};
use yggdryl::{MediaType, MimeType, Value, avro, json};

let schema = json::from_str(
    r#"{"type": "record", "name": "trade", "fields": [
        {"name": "symbol", "type": "string"},
        {"name": "quantity", "type": "long"},
        {"name": "price", "type": ["null", "double"], "default": null}
    ]}"#,
)?;
let rows = [
    json::from_str(r#"{"symbol": "AAPL", "quantity": 100, "price": 187.5}"#)?,
    json::from_str(r#"{"symbol": "MSFT", "quantity": 25, "price": null}"#)?,
];

let mut handle = Buffer::new();
handle.set_media_type(MediaType::new(MimeType::AVRO));
avro::write_container(&mut handle, &schema, &[("source", "docs")], &rows)?;

let container = avro::read_container(&handle)?;
assert_eq!(container.get("source"), Some("docs"));
assert_eq!(container.rows.len(), 2);
assert_eq!(
    container.rows[0].get_key_str("symbol").and_then(Value::as_str),
    Some("AAPL")
);
assert!(container.rows[1].get_key_str("price").is_some_and(Value::is_null));

## The container is the unit

In [ ]:
use yggdryl::io::{Buffer, IOBase};
use yggdryl::{avro, json};

let schema = json::from_str(
    r#"{"type": "record", "name": "row", "fields": [
        {"name": "id", "type": "long"}
    ]}"#,
)?;

// A header with no blocks is a complete, empty container.
let mut handle = Buffer::new();
avro::write_container(&mut handle, &schema, &[], &[])?;
assert!(avro::read_container(&handle)?.rows.is_empty());

// Bytes that are not a container say what was expected.
let mut wrong = Buffer::new();
wrong.write_all_bytes(b"not avro")?;
let message = avro::read_container(&wrong).unwrap_err().to_string();
assert!(message.contains("Avro object container"));